In [2]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterstats import zonal_stats
import glob
import pandas as pd
import os
from datetime import datetime
import re
import matplotlib.pyplot as plt
from exactextract import exact_extract

In [3]:
# OUTPUT_DIR_NAME = "Output"
# MERGED_OUTPUT_NAME = "wijk_alle_bomen_gecombineerd.gpkg"

# # Pad instellen
# # script_dir = os.path.dirname(os.path.abspath(__file__))
# output_dir = "Output\\"  # os.path.join(script_dir, OUTPUT_DIR_NAME)

# # Zoek alle .gpkg bestanden in de outputmap
# gpkg_files = glob.glob(os.path.join(output_dir, "schildersbuurt*.gpkg"))

# # Filter het doelbestand zelf eruit als dat er toevallig al staat
# gpkg_files = [f for f in gpkg_files if os.path.basename(f) != MERGED_OUTPUT_NAME]

# if not gpkg_files:
#     print(f"Geen .gpkg bestanden gevonden in: {output_dir}")
# else:
#     print(f"Gevonden: {len(gpkg_files)} GeoPackage bestanden om samen te voegen.")
    
#     gdf_list = []
#     for file_path in gpkg_files:
#         try:
#             print(f"Inlezen: {os.path.basename(file_path)}")
#             gdf = gpd.read_file(file_path)
#             gdf_list.append(gdf)
#         except Exception as e:
#             print(f"Fout bij inlezen van {os.path.basename(file_path)}: {e}")

#     if gdf_list:
#         # Combineer alle GeoDataFrames in één grote GeoDataFrame
#         print("Bezig met samenvoegen...")
#         merged_gdf = pd.concat(gdf_list, ignore_index=True)
#          #Remove duplicates based on geometry and attributes if necessary
#         initial_count = len(merged_gdf)
#         merged_gdf = merged_gdf.drop_duplicates(subset=['geometry'])
            
#         # Optie B (optioneel): Als polygonen net een mini-verschil hebben door tegelgrenzen,
#         # kun je op basis van de centroids (middelpunten) met een kleine tolerantie filteren:
#         merged_gdf['centroid'] = merged_gdf.geometry.centroid
#         merged_gdf = merged_gdf.drop_duplicates(subset=['centroid'])
#         merged_gdf = merged_gdf.drop(columns=['centroid'])
            
#         print(f"Dubbele bomen verwijderd: {initial_count - len(merged_gdf)} bomen samengevoegd/gewist.")
#         # Opslaan als één nieuw bestand
#         merged_output_path = os.path.join(output_dir, MERGED_OUTPUT_NAME)
#         merged_gdf.to_file(merged_output_path, driver="GPKG")
        
#         print("\n" + "="*40)
#         print(f" Klaar! Alle bestanden succesvol samengevoegd.")
#         print(f" Totaal aantal objecten (bomen): {len(merged_gdf)}")
#         print(f" Opgeslagen als: {merged_output_path}")
#         print("="*40)
#     else:
#         print("Geen geldige data gevonden om samen te voegen.")

In [6]:
grid_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\Schildersbuurt\\schildersbuurt_grid_extent.gpkg"
grid = gpd.read_file(grid_path)

grid["grid_id"] = grid.index

# ============================
# Add land cover features to the grid
# ============================

verharding = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\verharding_2024.tif"
onverhard = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\onverhard_2024.tif"
water = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\water_2024.tif"

with rasterio.open(verharding) as src:
    raster_crs = src.crs
if grid.crs != raster_crs:
    print("CRS of grid and raster do not match. Reprojecting grid to match raster CRS.")
    grid = grid.to_crs(raster_crs)

with rasterio.open(onverhard) as src:
    raster_crs = src.crs
if grid.crs != raster_crs:
    print("CRS of grid and raster do not match. Reprojecting grid to match raster CRS.")
    grid = grid.to_crs(raster_crs)

verharding_stats = exact_extract(verharding, grid, ['sum', 'count'], output = "pandas")
verharding_stats["percentage_verharding"] = np.round((verharding_stats["sum"] / verharding_stats["count"]) * 100, 2)

onverhard_stats = exact_extract(onverhard, grid, ['sum', 'count'], output = "pandas")
onverhard_stats["percentage_onverhard"] = np.round((onverhard_stats["sum"] / onverhard_stats["count"]) * 100, 2)

water_stats = exact_extract(water, grid, ['sum', 'count'], output = "pandas")
water_stats["percentage_water"] = np.round((water_stats["sum"] / water_stats["count"]) * 100, 2)

grid["percentage_verharding"] = verharding_stats["percentage_verharding"]
grid["percentage_onverhard"] = onverhard_stats["percentage_onverhard"]
grid["percentage_water"] = water_stats["percentage_water"]

print("Land cover features added to the grid.")


# ============================
# Add shadow features
# ============================

groene_schaduw = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\groene_schaduw_2025_2.tif"
grijze_schaduw = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\grijze_schaduw_2025_2.tif"
combined_schaduw = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\schaduwkaart.tif"

with rasterio.open(groene_schaduw) as src:
    raster_crs = src.crs
if grid.crs != raster_crs:
    print("CRS of grid and raster do not match. Reprojecting grid to match raster CRS.")
    grid = grid.to_crs(raster_crs)

green_shadow_stats = exact_extract(groene_schaduw, grid, ['sum', 'count'], output = "pandas")
green_shadow_stats["percentage_green_shadow"] = np.round((green_shadow_stats["sum"] / green_shadow_stats["count"]) * 100, 2)

grey_shadow_stats = exact_extract(grijze_schaduw, grid, ['sum', 'count'], output = "pandas")
grey_shadow_stats["percentage_grey_shadow"] = np.round((grey_shadow_stats["sum"] / grey_shadow_stats["count"]) * 100, 2)

combined_shadow_stats = exact_extract(combined_schaduw, grid, ['sum', 'count'], output = "pandas")
combined_shadow_stats["percentage_combined_shadow"] = np.round((combined_shadow_stats["sum"] / combined_shadow_stats["count"]) * 100, 2)

grid["percentage_green_shadow"] = green_shadow_stats["percentage_green_shadow"]
grid["percentage_grey_shadow"] = grey_shadow_stats["percentage_grey_shadow"]
grid["percentage_combined_shadow"] = combined_shadow_stats["percentage_combined_shadow"]

print("Shadow features added to the grid.")

# ============================
# Add tree features to grid
# ============================

trees = gpd.read_file("C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\Schildersbuurt\\alle_bomen_schildersbuurt.gpkg")

if trees.crs != grid.crs:
    trees = trees.to_crs(grid.crs)

trees["orig_area"] = trees.geometry.area

intersection = gpd.overlay(trees, grid, how='intersection')
intersection["clipped_area"] = intersection.geometry.area
intersection["area_ratio"] = intersection["clipped_area"] / intersection["orig_area"]

intersection["boomkroonvolume_m3"] = np.round(intersection["measured_cv_m3"] * intersection["area_ratio"], 2)
intersection["boomkroonoppervlakte_m2"] = np.round(intersection['measured_ca_m2'] * intersection['area_ratio'], 2)
intersection["biomassa_bermudez_kg"] = np.round(intersection['model_agb_kg_bermudez'] * intersection['area_ratio'], 2)

trees_sum = intersection.groupby("grid_id")[["boomkroonvolume_m3", "boomkroonoppervlakte_m2", "biomassa_bermudez_kg"]].sum().reset_index()

grid = grid.merge(trees_sum, on="grid_id", how="left")
grid["boomkroonvolume_m3"] = grid["boomkroonvolume_m3"].fillna(0)
grid["boomkroonoppervlakte_m2"] = grid["boomkroonoppervlakte_m2"].fillna(0)
grid["biomassa_bermudez_kg"] = grid["biomassa_bermudez_kg"].fillna(0)

print("Tree features added to the grid.")

# ============================
# Add ECOSTRESS temperature data to the grid
# ============================

temp_data_folder = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ECOSTRESS\\ECOSTRESS_sharp\\sharp\\"

tif_files = sorted(glob.glob(os.path.join(temp_data_folder, "*.tif")))

with rasterio.open(tif_files[0]) as src:
    raster_crs = src.crs

if grid.crs != raster_crs:
    print("CRS of grid and raster do not match. Reprojecting grid to match raster CRS.")
    grid = grid.to_crs(raster_crs)

centroids = [(geom.centroid.x, geom.centroid.y) for geom in grid.geometry]

for i, tif_path in enumerate(tif_files, 1):
    filename = os.path.basename(tif_path)

    # Bepaal kolomnaam op basis van datum/tijd (bijv. LST_20250602T135437)
    match = re.search(r"\d{8}T\d{6}", filename)
    if match:
        dt = datetime.strptime(match.group(0), "%Y%m%dT%H%M%S")
        col_name = f"LST_{dt.strftime('%Y-%m-%d %H:%M')}"
    else:
        col_name = f"LST_{i}"

    print(f"[{i}/{len(tif_files)}] Verwerken: {col_name}...")

    with rasterio.open(tif_path) as src:
        # Sample de pixelwaarden op de centroid-locaties
        sampled_values = [val[0] for val in src.sample(centroids)]
        nodata_val = src.nodata

        celsius_list = []
        for val in sampled_values:
            # Sla NoData / ongeldige waarden over
            if (
                val is None
                or (nodata_val is not None and val == nodata_val)
                or np.isnan(val)
            ):
                celsius_list.append(None)
            else:
                # Omrekenen van Kelvin naar Celsius
                celsius_list.append(round(val - 273.15, 2))

        # Voeg toe als nieuwe kolom aan het dataframe
        grid[col_name] = celsius_list      

print("ECOSTRESS temperature data added to the grid.")

# ============================
# Save the final grid with all features
# ============================

output_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\master_grid_with_features.gpkg"
grid.to_file(output_path, driver="GPKG")

print(f"Final grid with features saved to {output_path}")

CRS of grid and raster do not match. Reprojecting grid to match raster CRS.
Land cover features added to the grid.
CRS of grid and raster do not match. Reprojecting grid to match raster CRS.
Shadow features added to the grid.
Tree features added to the grid.
CRS of grid and raster do not match. Reprojecting grid to match raster CRS.
[1/31] Verwerken: LST_2025-06-02 13:54...
[2/31] Verwerken: LST_2025-06-02 15:30...
[3/31] Verwerken: LST_2025-06-03 13:05...
[4/31] Verwerken: LST_2025-06-04 12:18...
[5/31] Verwerken: LST_2025-06-05 13:06...
[6/31] Verwerken: LST_2025-06-06 12:17...
[7/31] Verwerken: LST_2025-06-09 11:29...
[8/31] Verwerken: LST_2025-06-11 11:29...
[9/31] Verwerken: LST_2025-06-12 09:04...
[10/31] Verwerken: LST_2025-06-15 09:51...
[11/31] Verwerken: LST_2025-06-16 09:03...
[12/31] Verwerken: LST_2025-06-18 07:26...
[13/31] Verwerken: LST_2025-06-29 04:58...
[14/31] Verwerken: LST_2025-06-30 04:10...
[15/31] Verwerken: LST_2025-07-10 22:26...
[16/31] Verwerken: LST_2025-0